<a href="https://colab.research.google.com/github/hungvuongminh-cell/AI-Testing-Learning/blob/main/Phase_01/0002_multi_llm_evaluator_groq_and_langchain_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain-groq

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Cấu hình API Key (Thay bằng key của bạn)
os.environ["GROQ_API_KEY"] = "groq_key"

# 2. Khai báo danh sách 5 model phổ biến trên Groq
MODELS = [
    "llama-3.3-70b-versatile",   # Index 0: Model mạnh nhất hiện tại của Meta
    "llama-3.1-8b-instant",      # Index 1: Model nhỏ, cực nhanh
    "gemma2-9b-it",              # Index 2: Model từ Google (bản mở)
    "mixtral-8x7b-32768",        # Index 3: Model từ Mistral AI
    "deepseek-r1-distill-llama-70b" # Index 4: Model chuyên suy luận (Reasoning)
]

# BIẾN ĐIỀU KHIỂN: Thay đổi số từ 0-4 để chọn model muốn test
MODEL_INDEX = 0

# 3. Khởi tạo Model
selected_model_name = MODELS[MODEL_INDEX]
model = ChatGroq(
    model=selected_model_name,
    temperature=0.1, # Để thấp để giảm hallucination (ảo tưởng)
    max_retries=2
)

# 4. Định nghĩa Prompt chuẩn (Structured Prompt)
system_template = """
- **Role**: Bạn là một chuyên gia Kiểm thử AI (AI Quality Assurance) với 10 năm kinh nghiệm.
- **Task**: Phân tích và đánh giá tính logic của đoạn văn bản được cung cấp.
- **Context**: Người dùng đang xây dựng bộ dataset chuẩn để đánh giá các LLM khác.
- **Instructions**:
    1. Trích xuất các thực thể chính.
    2. Chỉ ra các điểm mâu thuẫn nếu có.
    3. Đưa ra điểm tin cậy trên thang điểm 10.
- **Constraints (Avoiding Bias & Hallucination)**:
    - CHỈ dựa trên thông tin có trong văn bản được cung cấp.
    - Nếu thông tin không có, hãy trả lời "Không đủ dữ liệu", tuyệt đối không tự bịa ra thông tin.
    - Tránh các định kiến về giới tính, tôn giáo hoặc sắc tộc.
- **Response Style**: Trình bày dưới dạng Markdown, sử dụng bảng biểu cho các thông số kỹ thuật, ngôn ngữ chuyên nghiệp và súc tích.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", "{input_text}")
])

# 5. Thiết lập Chain và Chạy test
chain = prompt_template | model | StrOutputParser()

input_data = "Sản phẩm A có giá 100k, giảm giá 20% nhưng hóa đơn cuối cùng khách phải trả là 90k. Hãy phân tích tính logic."

print(f"--- ĐANG CHẠY TEST VỚI MODEL: {selected_model_name} ---\n")
response = chain.invoke({"input_text": input_data})

print(response)

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Cấu hình API Key
os.environ["GROQ_API_KEY"] = "groq_key"

# 2. Danh sách Model
MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "gemma2-9b-it",
    "mixtral-8x7b-32768",
    "deepseek-r1-distill-llama-70b"
]

MODEL_INDEX = 0

# 3. Khởi tạo Model
model = ChatGroq(model=MODELS[MODEL_INDEX], temperature=0.2)

# 4. MASTER PROMPT MẪU (Sức khỏe bình thường nhưng cấu trúc cực xịn)
# Đây là phần bạn có thể dùng để test khả năng tư duy của các model khác nhau.
master_prompt = """
- **Role**: Bạn là một Chuyên gia Tư vấn Sức khỏe Chủ động (Health Coach) với tư duy y khoa dựa trên bằng chứng (Evidence-based).

- **Task**: Xây dựng một lộ trình cải thiện chất lượng giấc ngủ và giảm căng thẳng cho nhân viên văn phòng.

- **Context**: Tôi là một lập trình viên 30 tuổi, thường xuyên làm việc với máy tính 10 tiếng/ngày, uống 3 ly cafe mỗi ngày và thường xuyên bị khó ngủ, tỉnh giấc lúc 3 giờ sáng. Tôi muốn cải thiện sức khỏe mà không dùng đến thuốc ngủ.

- **Instructions**:
    1. Giải thích cơ chế khoa học của việc tỉnh giấc giữa đêm dựa trên thói quen tiêu thụ caffeine.
    2. Đề xuất 3 bài tập vận động nhẹ nhàng tại chỗ có thể thực hiện trong giờ làm việc.
    3. Thiết kế quy trình "Sleep Hygiene" (vệ sinh giấc ngủ) trong 2 tiếng trước khi đi ngủ.

- **Constraints (Avoiding Bias & Hallucination)**:
    - Chỉ đưa ra các lời khuyên dựa trên các nghiên cứu y khoa đã được công nhận (như từ Mayo Clinic hoặc Harvard Health).
    - Tuyệt đối không được kê đơn thuốc hoặc chẩn đoán bệnh lý thay cho bác sĩ.
    - Nếu có phương pháp nào mang tính thực nghiệm, phải nêu rõ "Cần tham khảo ý kiến chuyên gia".
    - Không được tự bịa ra các số liệu thống kê không có thật.

- **Response Style**: Trình bày khoa học, dễ hiểu, sử dụng Bullet points để các bước hành động rõ ràng. Giọng văn khích lệ nhưng phải khách quan và nghiêm túc.
"""

# 5. Thiết lập Prompt Template (Dùng Master Prompt làm nội dung User gửi đi)
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Bạn là một trợ lý AI hữu ích."),
    ("user", master_prompt)
])

# 6. Chạy Chain
chain = prompt_template | model | StrOutputParser()

print(f"--- ĐANG TEST VỚI MODEL: {MODELS[MODEL_INDEX]} ---\n")
print(chain.invoke({}))

--- ĐANG TEST VỚI MODEL: llama-3.3-70b-versatile ---

Tôi hiểu rằng bạn đang gặp khó khăn với giấc ngủ và muốn cải thiện chất lượng giấc ngủ mà không dùng đến thuốc ngủ. Dưới đây là một số lời khuyên dựa trên các nghiên cứu y khoa đã được công nhận:

**Giải thích cơ chế khoa học của việc tỉnh giấc giữa đêm dựa trên thói quen tiêu thụ caffeine:**

Caffeine là một chất kích thích có thể ảnh hưởng đến giấc ngủ. Khi bạn tiêu thụ caffeine, nó sẽ được hấp thụ vào máu và ảnh hưởng đến hệ thống thần kinh trung ương. Caffeine có thể làm tăng mức độ alertness (tỉnh táo) và giảm mức độ mệt mỏi, nhưng nó cũng có thể gây ra khó ngủ và tỉnh giấc giữa đêm.

Theo nghiên cứu của Mayo Clinic, caffeine có thể ảnh hưởng đến giấc ngủ trong 4-6 giờ sau khi tiêu thụ. Điều này có nghĩa là nếu bạn uống 3 ly cafe vào buổi chiều, caffeine có thể vẫn còn trong máu của bạn vào buổi tối và gây ra khó ngủ.

**Đề xuất 3 bài tập vận động nhẹ nhàng tại chỗ có thể thực hiện trong giờ làm việc:**

Dưới đây là 3 bài tập v